In [1]:
!pip install xgboost

import re, math, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as spstats
from scipy.stats import randint, uniform

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, rdMolDescriptors, MolToSmiles, MolFromSmiles
from rdkit.Chem import rdFingerprintGenerator

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, IsolationForest
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import pairwise_distances
from sklearn.impute import SimpleImputer
from numpy.linalg import pinv
from xgboost import XGBRegressor

Defaulting to user installation because normal site-packages is not writeable


In [9]:

# Dataset
RAW = [
  ['TNT','Cc1c([N+](=O)[O-])cc([N+](=O)[O-])cc1[N+](=O)[O-]','C7H5N3O6',227.13,1.654,15.0,300,6900,-54.4],
  ['RDX','O=[N+]([O-])N1CN([N+](=O)[O-])CN([N+](=O)[O-])C1','C3H6N6O6',222.12,1.820,7.4,210,8750,86.3],
  ['HMX','O=[N+]([O-])N1CN([N+](=O)[O-])CN([N+](=O)[O-])CN([N+](=O)[O-])C1','C4H8N8O8',296.16,1.905,7.4,280,9110,75.0],
  ['PETN','C(CO[N+](=O)[O-])(CO[N+](=O)[O-])(CO[N+](=O)[O-])CO[N+](=O)[O-]','C5H8N4O12',316.14,1.778,3.0,160,8400,-538.5],
  ['TATB','Nc1c([N+](=O)[O-])c(N)c([N+](=O)[O-])c(N)c1[N+](=O)[O-]','C6H6N6O6',258.15,1.939,50.0,360,7350,-139.7],
  ['NTO','O=[N+]([O-])c1n[nH]c(=O)[nH]1','C2H2N4O3',130.06,1.930,71.0,273,8000,-91.7],
  ['FOX-7','NC(=C([N+](=O)[O-])[N+](=O)[O-])N','C2H4N4O4',148.08,1.885,25.0,230,8870,-133.8],
  ['DNAN','COc1ccc([N+](=O)[O-])cc1[N+](=O)[O-]','C7H6N2O5',198.13,1.321,40.0,190,6900,-182.7],
  ['CL-20','O=[N+]([O-])N1CN2CN([N+](=O)[O-])CN1[N+](=O)[O-]','C6H6N12O12',438.19,2.044,4.0,230,9400,460.0],
  ['Tetryl','CN([N+](=O)[O-])c1c([N+](=O)[O-])cc([N+](=O)[O-])cc1[N+](=O)[O-]','C7H5N5O8',287.14,1.730,3.0,195,7850,18.9],
  ['TNAZ','O=[N+]([O-])C1(CN([N+](=O)[O-])C1[N+](=O)[O-])[N+](=O)[O-]','C3H4N4O8',240.08,1.840,12.0,200,8740,17.0],
  ['LLM-105','Nc1nc2c([N+](=O)[O-])n[nH]c2c([N+](=O)[O-])n1','C4H4N8O4',228.12,1.913,40.0,328,8560,135.6],
  ['ANTA','Nc1nnc([N+](=O)[O-])[nH]1','C2H3N5O2',129.08,1.840,60.0,260,8200,28.0],
  ['NQ','NC(=N)N[N+](=O)[O-]','CH4N4O2',104.07,1.710,95.0,245,7900,-92.5],
  ['Picric','Oc1c([N+](=O)[O-])cc([N+](=O)[O-])cc1[N+](=O)[O-]','C6H3N3O7',229.10,1.763,7.4,300,7350,-217.8],
  ['DATB','Nc1c([N+](=O)[O-])c(N)c([N+](=O)[O-])cc1[N+](=O)[O-]','C6H5N5O6',243.13,1.840,50.0,320,7520,-143.5],
  ['TETNB','O=[N+]([O-])c1c([N+](=O)[O-])c([N+](=O)[O-])cc([N+](=O)[O-])c1','C6H2N4O8',258.10,1.814,8.0,180,7900,125.0],
  ['HNS','O=[N+]([O-])c1ccc(C=Cc2ccc([N+](=O)[O-])cc2[N+](=O)[O-])c([N+](=O)[O-])c1','C14H6N6O12',450.22,1.745,7.0,318,7600,22.5],
  ['NG','O[C@@H](CO[N+](=O)[O-])CO[N+](=O)[O-]','C3H5N3O9',227.09,1.591,0.2,200,7700,-370.9],
  ['EDNA','O=[N+]([O-])NCC[N+](=O)[O-]','C2H5N3O4',135.08,1.713,5.0,175,7570,-148.0],
  ['BTNEU','O=[N+]([O-])c1c([N+](=O)[O-])c([N+](=O)[O-])c([N+](=O)[O-])c([N+](=O)[O-])c1[N+](=O)[O-]','C6N6O12',348.10,1.980,8.0,230,9000,385.0],
  ['MHN','OC(CO[N+](=O)[O-])CO[N+](=O)[O-]','C3H6N2O7',182.09,1.450,60.0,175,6500,-403.0],
  ['NONA','O=[N+]([O-])c1c([N+](=O)[O-])c([N+](=O)[O-])c([N+](=O)[O-])c([N+](=O)[O-])c1[N+](=O)[O-]','C6HN5O10',287.10,1.870,6.0,220,8700,300.0],
  ['Heptyl','CC(C)(C)c1c([N+](=O)[O-])cc([N+](=O)[O-])cc1[N+](=O)[O-]','C10H11N3O6',269.21,1.490,20.0,190,6800,-136.0],
  ['DNAZ','O=[N+]([O-])N1CC([N+](=O)[O-])C1','C3H5N3O4',147.09,1.699,14.0,155,7550,-148.0],
  ['BTF','O=[N+]([O-])c1c([N+](=O)[O-])c([N+](=O)[O-])c([N+](=O)[O-])c([N+](=O)[O-])n1[N+](=O)[O-]','C5N6O12',334.10,1.920,5.0,210,8900,420.0],
  ['AN','N.[N+](=O)([O-])[O-]','H4N2O3',80.04,1.720,100.0,230,4500,-365.6],
  ['AP','[NH4+].[O-]Cl(=O)(=O)=O','ClH4NO4',117.49,1.950,110.0,300,5500,-295.8],
  ['Nitroguanidine','NC(=N)N[N+](=O)[O-]','CH4N4O2',104.07,1.770,95.0,250,7900,-92.5],
  ['DATB-2','Nc1c([N+](=O)[O-])cc([N+](=O)[O-])cc1[N+](=O)[O-]','C6H5N4O6',229.12,1.797,25.0,295,7480,-120.0],
]

COLS = ['Name','SMILES','Formula','MW','Density','IS_J','Td_C','Dv_ms','Hf_kJmol']
df_raw = pd.DataFrame(RAW, columns=COLS)

def parse_formula(formula):
    counts = {}
    for elem in ['C','H','N','O']:
        m = re.search(rf'{elem}(\d*)', str(formula))
        counts[elem] = int(m.group(1)) if m and m.group(1) else (1 if m else 0)
    return counts

fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=256)

def get_descriptors(row):
    mol = MolFromSmiles(row['SMILES'])
    if mol is None: return None
    e = parse_formula(row['Formula'])
    mw = float(row['MW'])
    ob = (1600/mw)*(2*e['C']+e['H']/2-e['O'])
    fp = fpgen.GetFingerprint(mol)
    arr = np.zeros(256, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    d = {
        'MW': mw, 'nN': e['N'], 'nO': e['O'], 'nC': e['C'],
        'nNO2': row['SMILES'].count('[N+](=O)[O-]'),
        'NC': round(e['N']/(e['C']+1e-9), 3), 'OB': round(ob, 2),
        'TPSA': round(Descriptors.TPSA(mol), 2),
        'LogP': round(Descriptors.MolLogP(mol), 3),
        'HBD': rdMolDescriptors.CalcNumHBD(mol),
        'HBA': rdMolDescriptors.CalcNumHBA(mol),
        'Rings': rdMolDescriptors.CalcNumRings(mol),
        'ArR': rdMolDescriptors.CalcNumAromaticRings(mol),
        'RotB': rdMolDescriptors.CalcNumRotatableBonds(mol),
        'FrCSP3': round(Descriptors.FractionCSP3(mol), 4),
        'Chi0': round(Descriptors.Chi0(mol), 4),
        'Kappa1': round(Descriptors.Kappa1(mol), 4),
        'BalabanJ': round(Descriptors.BalabanJ(mol), 4),
    }
    for i, b in enumerate(arr): d[f'FP{i}'] = int(b)
    return d

# --- 1. Generate Descriptors & Track Valid Rows ---
desc_list = [get_descriptors(row) for _, row in df_raw.iterrows()]

# Create the mask to keep only rows where get_descriptors worked
valid_rows_mask = [bool(d) for d in desc_list]

# Align both DataFrames to the exact same 28 valid rows right here!
df_desc = pd.DataFrame([d for d in desc_list if d])
df_meta = df_raw.loc[valid_rows_mask, [c for c in COLS if c not in ['SMILES','Formula']]].reset_index(drop=True)

# --- 2. Feature Selection & Preprocessing (Now running on aligned 28 rows) ---
vt = VarianceThreshold(0.01)
X_vt = vt.fit_transform(df_desc)
feat_names = df_desc.columns[vt.get_support()].tolist()
X_df = pd.DataFrame(X_vt, columns=feat_names)

corr_mat = X_df.corr().abs()
upper    = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
to_drop  = [c for c in upper.columns if any(upper[c] > 0.92)]
X_clean  = X_df.drop(columns=to_drop)

# --- 3. Scaling (Will correctly output shape: [28, num_features]) ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)
feat_names_final = X_clean.columns.tolist()

# --- 4. Pull Out Aligned Target Arrays ---
y_IS  = df_meta['IS_J'].astype(float).values
y_rho = df_meta['Density'].astype(float).values
y_Dv  = df_meta['Dv_ms'].astype(float).values
y_Td  = df_meta['Td_C'].astype(float).values
y_Hf  = df_meta['Hf_kJmol'].astype(float).values
names = df_meta['Name'].values

# --- 5. Verify and Split ---
print(f"X_scaled shape: {X_scaled.shape}") # Expected: (28, n)
print(f"y_IS shape: {y_IS.shape}")         # Expected: (28,)
print(f"names shape: {names.shape}")       # Expected: (28,)

# This will now execute perfectly without errors!
X_tr, X_te, y_tr, y_te, n_tr, n_te = train_test_split(
    X_scaled, y_IS, names, test_size=0.30, random_state=42
)
def evaluate(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    p_tr = model.predict(Xtr)
    p_te = model.predict(Xte)
    return {
        'Model': name,
        'R2_train': round(r2_score(ytr, p_tr), 3),
        'R2_test':  round(r2_score(yte, p_te), 3),
        'MAE_test': round(mean_absolute_error(yte, p_te), 2),
        'RMSE_test':round(np.sqrt(mean_squared_error(yte, p_te)), 2),
    }, model.predict(Xte)

print("B0 ✓ dataset ready:", X_scaled.shape)





X_scaled shape: (28, 83)
y_IS shape: (28,)
names shape: (28,)
B0 ✓ dataset ready: (28, 83)


[14:46:53] SMILES Parse Error: unclosed ring for input: 'O=[N+]([O-])N1CN2CN([N+](=O)[O-])CN1[N+](=O)[O-]'
[14:46:54] Can't kekulize mol.  Unkekulized atoms: 3 4 8 12 16


In [14]:
# Block 1 - MLR
struct_feats = ['MW','nN','nO','nC','nNO2','NC','OB','TPSA','LogP','HBD','HBA','Rings','ArR','RotB','FrCSP3','Chi0','Kappa1']
struct_feats = [f for f in struct_feats if f in X_clean.columns]
X_struct = scaler.fit_transform(X_clean[struct_feats])
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(X_struct, y_IS, test_size=0.30, random_state=42)

for name, model in [('MLR', LinearRegression()), 
                    ('Ridge', Ridge(alpha=1.0, solver='svd')), 
                    ('Lasso', Lasso(alpha=0.5, max_iter=5000))]:
    res, _ = evaluate(name, model, Xs_tr, Xs_te, ys_tr, ys_te)
    print(f"B1 {res['Model']}: R2_test={res['R2_test']}")


B1 MLR: R2_test=-29.8
B1 Ridge: R2_test=0.711
B1 Lasso: R2_test=0.624


In [15]:
# Block 2 - PLS
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = [cross_val_score(PLSRegression(n_components=n), X_struct, y_IS, cv=cv, scoring='r2').mean()
         for n in range(1, min(8, len(struct_feats)))]
best_n = range(1, min(8, len(struct_feats)))[int(np.argmax(cv_r2))]
res_pls, _ = evaluate('PLS', PLSRegression(n_components=best_n), Xs_tr, Xs_te, ys_tr, ys_te)
print(f"B2 PLS n={best_n}: R2_test={res_pls['R2_test']}")


B2 PLS n=4: R2_test=0.654


In [16]:
# Block 3 - SVM
for kernel in ['linear', 'rbf']:
    res, _ = evaluate(f'SVM({kernel})', SVR(kernel=kernel, C=10, gamma='scale'), X_tr, X_te, y_tr, y_te)
    print(f"B3 {res['Model']}: R2_test={res['R2_test']}")


B3 SVM(linear): R2_test=0.535
B3 SVM(rbf): R2_test=0.04


In [17]:
# Block 4 - RF
rf = RandomForestRegressor(n_estimators=100, random_state=42)
res_rf, _ = evaluate('RF', rf, X_tr, X_te, y_tr, y_te)
print(f"B4 RF: R2_test={res_rf['R2_test']}")
print(f"   top feature: {feat_names_final[np.argmax(rf.feature_importances_)]}")


B4 RF: R2_test=0.801
   top feature: RotB


In [18]:
# Block 5 - GradBoost + XGBoost
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
res_gb, _ = evaluate('GB', gb, X_tr, X_te, y_tr, y_te)
print(f"B5 GB: R2_test={res_gb['R2_test']}")
xgb = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42, verbosity=0)
res_xgb, _ = evaluate('XGB', xgb, X_tr, X_te, y_tr, y_te)
print(f"B5 XGB: R2_test={res_xgb['R2_test']}")


B5 GB: R2_test=0.807
B5 XGB: R2_test=0.865


In [19]:
# Block 6 - CV
cv5 = cross_val_score(RandomForestRegressor(n_estimators=100, random_state=42), X_scaled, y_IS, cv=5, scoring='r2')
print(f"B6 RF 5-fold CV: {cv5.mean():.3f} ± {cv5.std():.3f}")


B6 RF 5-fold CV: 0.191 ± 0.816


In [20]:
# Block 7 - Grid + Random search (fast)
gs = GridSearchCV(SVR(), {'C':[1,10],'gamma':['scale'],'kernel':['rbf']}, cv=3, scoring='r2')
gs.fit(X_scaled, y_IS)
print(f"B7 GridSearch best: {gs.best_params_}, R2={gs.best_score_:.3f}")
rs = RandomizedSearchCV(RandomForestRegressor(random_state=42), {'n_estimators': randint(50,200), 'max_depth':[3,5,None]},
    n_iter=5, cv=3, scoring='r2', random_state=42)
rs.fit(X_scaled, y_IS)
print(f"B7 RandomSearch best R2: {rs.best_score_:.3f}")


B7 GridSearch best: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}, R2=-0.090
B7 RandomSearch best R2: 0.425


In [21]:
# Block 9 - All properties
for label, y_prop in [('IS',y_IS),('rho',y_rho),('Dv',y_Dv)]:
    Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(X_scaled, y_prop, test_size=0.30, random_state=42)
    rf_p = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_p.fit(Xp_tr, yp_tr)
    print(f"B9 RF {label}: R2_test={r2_score(yp_te, rf_p.predict(Xp_te)):.3f}")


B9 RF IS: R2_test=0.801
B9 RF rho: R2_test=0.234
B9 RF Dv: R2_test=0.729


In [22]:
# Block 10 - DoA
struct_idx = [i for i, f in enumerate(feat_names_final) if not f.startswith('FP')]
Xs_tr_doa = X_tr[:, struct_idx]
Xs_te_doa = X_te[:, struct_idx]
h_star = 3 * (Xs_tr_doa.shape[1]+1) / Xs_tr_doa.shape[0]
h_test = np.array([float(x @ pinv(Xs_tr_doa.T @ Xs_tr_doa) @ x) for x in Xs_te_doa])
rf_doa = RandomForestRegressor(n_estimators=100, random_state=42)
rf_doa.fit(X_tr, y_tr)
pred_doa = rf_doa.predict(X_te)
dist_mat = pairwise_distances(X_te, X_tr)
min_dist = dist_mat.min(axis=1)
print(f"B10 DoA: h*={h_star:.3f}, outside={sum(h_test>h_star)}, min_dist range=[{min_dist.min():.2f},{min_dist.max():.2f}]")
tree_preds = np.array([t.predict(X_te) for t in rf_doa.estimators_])
print(f"B10 Tree std range: [{tree_preds.std(axis=0).min():.2f}, {tree_preds.std(axis=0).max():.2f}]")


B10 DoA: h*=2.526, outside=6, min_dist range=[0.00,11.45]
B10 Tree std range: [9.42, 34.69]
